# HiSparse 入门 (6/8)：一步前向怎么走 —— decode / prefill / 混合批次

> **一句话结论**：decode 走"解析→补缺→直读"，prefill 走"暂存计划→搬入行→组装→注意力"，混合批次两条路径并行、输出拼接。

## 1. Decode 路径

1. `prepare_for_batch`：按批次准备 resolver 输入；
2. swap-in 解析：resolver 把逻辑 token 翻译成物理行，产出搬移与淘汰计划；
3. 缺失行 gather 到 Resident / Hot；
4. 生成物理索引，SFA 在热区原地读取 656B 行做注意力。

## 2. Prefill 路径

1. 生成 staging 暂存计划；
2. `stage_prefill_rows`：把 prefill 行搬入 Resident；
3. pack 组装标准 656B 行；
4. 注意力读取。C8 全驻留时可走分页捷径。

## 3. 混合批次

同一批里 decode 请求与 prefill 请求并存：

- decode 行走解析路径；
- prefill 行走暂存路径；
- 两路输出在注意力层拼接成批次结果。

## 4. 写入与 Host 镜像

- `write_target()`：把新 KV 行 scatter 写回 Resident / Hot；
- Connector 把 Resident 行镜像回 Host 池，保证 Host 留底完整；
- 捕获期写回延迟发布；draft 与最终请求重叠时做围栏判定；
- 分组回收时按组失效旧槽。

## 5. 小结

- decode：解析 → 补缺 → 直读；
- prefill：暂存 → 搬入 → 组装 → 注意力；
- 写入后镜像回 Host，捕获期遵守围栏与延迟发布；
- 下一篇看后端在 vllm-ascend 里的接线与模块分工。

**下一篇**：`07-backend-wiring.ipynb`
